In [0]:
%python
from pyspark.sql.functions import *

# Define catalog variable for parameterization
catalog_name = "smartretail_corp"

# Silver Layer (Incremental Processing)

Initial Full Load

In [0]:
drop table if exists smartretail_corp.silver.customers;

create table if not exists smartretail_corp.silver.customers (
    customer_id int primary key,
    name string,
    city string,
    state string,
    signup_date date,
    created_at timestamp,
    updated_at timestamp,
    record_created_at timestamp,
    record_updated_at timestamp
  );

drop table if exists smartretail_corp.silver.order_items;

create table if not exists smartretail_corp.silver.order_items (
    order_item_id int primary key,
    order_id int,
    product_id int,
    quantity int,
    price float,
    created_at timestamp,
    updated_at timestamp,
    record_created_at timestamp,
    record_updated_at timestamp
  );

drop table if exists smartretail_corp.silver.orders;

create table if not exists smartretail_corp.silver.orders (
    order_id int primary key,
    customer_id int,
    order_date date,
    total_amount float,
    order_status string,
    created_at timestamp,
    updated_at timestamp,
    record_created_at timestamp,
    record_updated_at timestamp
  );

drop table if exists smartretail_corp.silver.products;

create table if not exists smartretail_corp.silver.products (
    product_id int primary key,
    product_name string,
    category string,
    price float,
    created_at timestamp,
    updated_at timestamp,
    record_created_at timestamp,
    record_updated_at timestamp
  );

-- Control table to track last run time per table
drop table if exists smartretail_corp.silver.table_logger;

create table if not exists smartretail_corp.silver.table_logger (
    table_name string primary key,
    last_run_time timestamp
  );

In [0]:
insert into smartretail_corp.silver.customers (
  customer_id, name, city, state, signup_date, created_at, updated_at, record_created_at
)
  select
    cast(customer_id as int) as customer_id,
    name,
    city,
    state,
    cast(signup_date as date) as signup_date,
    created_at,
    updated_at,
    current_timestamp() record_created_at
  from
    smartretail_corp.bronze.raw_customers;

insert into smartretail_corp.silver.table_logger
  select
    'customers',
    max(record_created_at) as last_run
  from
    smartretail_corp.silver.customers;

num_affected_rows,num_inserted_rows
1,1


In [0]:
insert into smartretail_corp.silver.order_items (
  order_item_id, order_id, product_id, quantity, price, created_at, updated_at, record_created_at
)
  select
    cast(order_item_id as int) as order_item_id,
    cast(order_id as int) as order_id,
    cast(product_id as int) as product_id,
    cast(quantity as int) as quantity,
    cast(price as float) as price,
    created_at,
    updated_at,
    current_timestamp() record_created_at
  from
    smartretail_corp.bronze.raw_order_items;

insert into smartretail_corp.silver.table_logger
  select
    'order_items',
    max(record_created_at) as last_run
  from
    smartretail_corp.silver.order_items;

num_affected_rows,num_inserted_rows
1,1


In [0]:
insert into smartretail_corp.silver.orders (
  order_id, customer_id, order_date, total_amount, order_status, created_at, updated_at, record_created_at
)
  select
    cast(order_id as int) as order_id,
    cast(customer_id as int) as customer_id,
    cast(order_date as date) as order_date,
    cast(total_amount as float) as total_amount,
    order_status,
    created_at,
    updated_at,
    current_timestamp () record_created_at
  from
    smartretail_corp.bronze.raw_orders;

insert into smartretail_corp.silver.table_logger
  select
    'orders',
    max(record_created_at) as last_run
  from
    smartretail_corp.silver.orders;

num_affected_rows,num_inserted_rows
1,1


In [0]:
insert into smartretail_corp.silver.products (
  product_id, product_name, category, price, created_at, updated_at, record_created_at
)
  select
    cast(product_id as int) as product_id,
    product_name,
    category,
    cast(price as float) as price,
    created_at,
    updated_at,
    current_timestamp() record_created_at
  from
    smartretail_corp.bronze.raw_products;

insert into smartretail_corp.silver.table_logger
  select
    'products',
    max(record_created_at) as last_run
  from
    smartretail_corp.silver.products;

num_affected_rows,num_inserted_rows
1,1


Incremental Load with MERGE (Upsert)

In [0]:
-- Get last run time for customers
set var last_run = (
  select
    last_run_time
  from
    smartretail_corp.silver.table_logger
  where
    table_name = 'customers'
);

-- Merge customers with incremental data
merge into
  smartretail_corp.silver.customers as target
using (
  select
    cast(customer_id as int) as customer_id,
    name,
    city,
    state,
    cast(signup_date as date) as signup_date,
    created_at,
    updated_at
  from
    smartretail_corp.bronze.raw_customers
  where
    created_at > last_run
    or updated_at > last_run
) as source
on
  target.customer_id = source.customer_id
when matched then update set
  target.name = source.name,
  target.city = source.city,
  target.state = source.state,
  target.signup_date = source.signup_date,
  target.created_at = source.created_at,
  target.updated_at = source.updated_at,
  target.record_updated_at = current_timestamp()
when not matched then insert (
    customer_id, name, city, state, signup_date, created_at, updated_at, record_created_at
  )
  values (
    source.customer_id,
    source.name,
    source.city,
    source.state,
    source.signup_date,
    source.created_at,
    source.updated_at,
    current_timestamp()
  );

-- Update control table
update
  smartretail_corp.silver.table_logger
set
  last_run_time = current_timestamp()
where
  table_name = 'customers';

num_affected_rows
1


In [0]:
-- Get last run time for order_items
set var last_run = (
  select
    last_run_time
  from
    smartretail_corp.silver.table_logger
  where
    table_name = 'order_items'
);

-- Merge order_items with incremental data
merge into
  smartretail_corp.silver.order_items as target
using (
  select
    cast(order_item_id as int) as order_item_id,
    cast(order_id as int) as order_id,
    cast(product_id as int) as product_id,
    cast(quantity as int) as quantity,
    cast(price as float) as price,
    created_at,
    updated_at
  from
    smartretail_corp.bronze.raw_order_items
  where
    created_at > last_run
    or updated_at > last_run
) as source
on
  target.order_item_id = source.order_item_id
when matched then update set
  target.order_id = source.order_id,
  target.product_id = source.product_id,
  target.quantity = source.quantity,
  target.price = source.price,
  target.created_at = source.created_at,
  target.updated_at = source.updated_at,
  target.record_updated_at = current_timestamp()
when not matched then insert (
    order_item_id, order_id, product_id, quantity, price, created_at, updated_at, record_created_at
  )
  values (
    source.order_item_id,
    source.order_id,
    source.order_id,
    source.quantity,
    source.price,
    source.created_at,
    source.updated_at,
    current_timestamp()
  );

-- Update control table
update
  smartretail_corp.silver.table_logger
set
  last_run_time = current_timestamp()
where
  table_name = 'order_items';

num_affected_rows
1


In [0]:
-- Get last run time for orders
set var last_run = (
  select
    last_run_time
  from
    smartretail_corp.silver.table_logger
  where
    table_name = 'orders'
);

-- Merge orders with incremental data
merge into
  smartretail_corp.silver.orders as target
using (
  select
    cast(order_id as int) as order_id,
    cast(customer_id as int) as customer_id,
    cast(order_date as date) as order_date,
    cast(total_amount as float) as total_amount,
    order_status,
    created_at,
    updated_at
  from
    smartretail_corp.bronze.raw_orders
  where
    created_at > last_run
    or updated_at > last_run
) as source
on
  target.order_id = source.order_id
when matched then update set
  target.customer_id = source.customer_id,
  target.order_date = source.order_date,
  target.total_amount = source.total_amount,
  target.order_status = source.order_status,
  target.created_at = source.created_at,
  target.updated_at = source.updated_at,
  target.record_updated_at = current_timestamp()
when not matched then insert (
    order_id,
    customer_id,
    order_date,
    total_amount,
    order_status,
    created_at,
    updated_at,
    record_created_at
  )
  values (
    source.order_id,
    source.customer_id,
    source.order_date,
    source.total_amount,
    source.order_status,
    source.created_at,
    source.updated_at,
    current_timestamp()
  );

-- Update control table
update
  smartretail_corp.silver.table_logger
set
  last_run_time = current_timestamp()
where
  table_name = 'orders';

num_affected_rows
1


In [0]:
-- Get last run time for products
set var last_run = (
  select
    last_run_time
  from
    smartretail_corp.silver.table_logger
  where
    table_name = 'products'
);

-- Merge products with incremental data
merge into
  smartretail_corp.silver.products as target
using (
  select
    cast(product_id as int) as product_id,
    product_name,
    category,
    cast(price as float) as price,
    created_at,
    updated_at
  from
    smartretail_corp.bronze.raw_products
  where
    created_at > last_run
    or updated_at > last_run
) as source
on
  target.product_id = source.product_id
when matched then update set
  target.product_name = source.product_name,
  target.category = source.category,
  target.price = source.price,
  target.created_at = source.created_at,
  target.updated_at = source.updated_at,
  target.record_updated_at = current_timestamp()
when not matched then insert (
    product_id, product_name, category, price, created_at, updated_at, record_created_at
  )
  values (
    source.product_id,
    source.product_name,
    source.category,
    source.price,
    source.created_at,
    source.updated_at,
    current_timestamp()
  );

-- Update control table
update
  smartretail_corp.silver.table_logger
set
  last_run_time = current_timestamp()
where
  table_name = 'products';

num_affected_rows
1


Table Joins

In [0]:
-- Join orders with customers to get customer information
drop table if exists smartretail_corp.silver.orders_with_customers;
create table smartretail_corp.silver.orders_with_customers as 
select
  o.order_id,
  o.order_date,
  o.total_amount,
  o.order_status,
  c.customer_id,
  c.name as customer_name,
  c.city,
  c.state,
  c.signup_date
from
  smartretail_corp.silver.orders o
  inner join smartretail_corp.silver.customers c on o.customer_id = c.customer_id;

num_affected_rows,num_inserted_rows


In [0]:
-- Join orders with order_items and products to get product information
drop table if exists smartretail_corp.silver.orders_with_products;
create table smartretail_corp.silver.orders_with_products as
select
  o.order_id,
  o.order_date,
  o.order_status,
  o.customer_id,
  oi.order_item_id,
  p.product_id,
  p.product_name,
  p.category,
  oi.quantity,
  oi.price as item_price,
  (oi.quantity * oi.price) as line_total
from
  smartretail_corp.silver.orders o
  inner join smartretail_corp.silver.order_items oi on o.order_id = oi.order_id
  inner join smartretail_corp.silver.products p on oi.product_id = p.product_id;

num_affected_rows,num_inserted_rows
